<h1> Sales and Customer Data </h1>

<h3> By: Ryan Moore </h3>

In [52]:
import sqlite3
import pandas as pd

In [54]:
#Load the dataset in to a database
df = pd.read_csv('/Users/Ryan/Code/Portfolio/Customer_data_SQL/customer.csv', header = 0)
conn = sqlite3.connect('customer.db')
df.to_sql('customer', conn, if_exists='replace', index=False)
conn.close()

In [57]:
#Load first 10 lines to ensure data is loaded correctly
conn = sqlite3.connect('customer.db')
sql_head = """
SELECT *
FROM customer
LIMIT 10;"""

sql_header = pd.read_sql_query(sql_head, conn)
conn.close()
print(sql_header)

  customer_id  gender   age payment_method
0     C241288  Female  28.0    Credit Card
1     C111565    Male  21.0     Debit Card
2     C266599    Male  20.0           Cash
3     C988172  Female  66.0    Credit Card
4     C189076  Female  53.0           Cash
5     C657758  Female  28.0    Credit Card
6     C151197  Female  49.0           Cash
7     C176086  Female  32.0    Credit Card
8     C159642    Male  69.0    Credit Card
9     C283361  Female  60.0    Credit Card


The dataset has customer id, the gender, age, and the payment method. 

In [58]:
#Find out how many transactions are in the dataset
conn = sqlite3.connect('customer.db')
sql_query = """
SELECT COUNT(*)
FROM customer;"""

sql_query = pd.read_sql_query(sql_query, conn)
conn.close()
print(sql_query)

   COUNT(*)
0     99457


There is a total of 99,457 transactions.  

In [59]:
#Find out the most common type of payment method
conn = sqlite3.connect('customer.db')
sql_query = """
SELECT payment_method, COUNT(*) AS Count
FROM customer
GROUP BY payment_method
ORDER BY Count;"""

sql_query = pd.read_sql_query(sql_query, conn)
conn.close()
print(sql_query)

  payment_method  Count
0     Debit Card  20079
1    Credit Card  34931
2           Cash  44447


In [60]:
#Find out the percentage of each transaction
conn = sqlite3.connect('customer.db')
sql_query = """
SELECT payment_method, ROUND((CAST(COUNT(*) AS REAL) / 99457)*100, 2) AS Percentage
FROM customer
GROUP BY payment_method
ORDER BY Percentage;"""

sql_query = pd.read_sql_query(sql_query, conn)
conn.close()
print(sql_query)

  payment_method  Percentage
0     Debit Card       20.19
1    Credit Card       35.12
2           Cash       44.69


The most commonly use payment method was cash, with the least being debit card. 

45% of transactions were done with cash
35% of transactions were done with credit card
20% of transactions were done with debit card.

This is suprising as typically cash transactions are not that high. 

In [70]:
#Find out the what customer placed the most orders
conn = sqlite3.connect('customer.db')
sql_query = """
SELECT customer_id, COUNT(*) AS Num_Orders
FROM customer
GROUP BY customer_id
ORDER BY Num_Orders DESC;"""

sql_query = pd.read_sql_query(sql_query, conn)
conn.close()
print(sql_query)

      customer_id  Num_Orders
0         C999995           1
1         C999976           1
2         C999974           1
3         C999910           1
4         C999886           1
...           ...         ...
99452     C100019           1
99453     C100012           1
99454     C100006           1
99455     C100005           1
99456     C100004           1

[99457 rows x 2 columns]


All of the customer ids are unique, this means that the customers are not returning or that the company is not tracking the returning customers. 

In [79]:
#Find out the what age placed the most orders
conn = sqlite3.connect('customer.db')
sql_query = """
SELECT 
    CASE
        WHEN age BETWEEN 0 AND 19 THEN '0-19'
        WHEN age BETWEEN 20 AND 29 THEN '20-29'
        WHEN age BETWEEN 30 AND 39 THEN '30-39'
        WHEN age BETWEEN 40 AND 49 THEN '40-49'
        WHEN age BETWEEN 50 AND 59 THEN '50-59'
        ELSE '60+'
    END AS Age_Bracket,
    COUNT(*) AS Total_Customers
FROM customer
GROUP BY Age_Bracket
;"""

sql_query = pd.read_sql_query(sql_query, conn)
conn.close()
print(sql_query)

  Age_Bracket  Total_Customers
0        0-19             3773
1       20-29            19244
2       30-39            19267
3       40-49            19129
4       50-59            18907
5         60+            19137


The customers ages are evenly spread out from 20 - 60+.  This means that customers of all ages are making purchases. 

In [86]:
#Find out the what gender placed the most orders
conn = sqlite3.connect('customer.db')
sql_query = """
SELECT gender, COUNT(*) AS Num_Orders
FROM customer
GROUP BY gender
ORDER BY Num_Orders DESC;"""

sql_query = pd.read_sql_query(sql_query, conn)
conn.close()
print(sql_query)

   gender  Num_Orders
0  Female       59482
1    Male       39975


In [88]:
#Find out the what gender placed the most orders
conn = sqlite3.connect('customer.db')
sql_query = """
SELECT gender, ROUND((CAST(COUNT(*) AS REAL) / 99457)*100, 2) AS Percentage
FROM customer
GROUP BY gender
;"""

sql_query = pd.read_sql_query(sql_query, conn)
conn.close()
print(sql_query)

   gender  Percentage
0  Female       59.81
1    Male       40.19


There is slightly more females that are purchasing with nearly 60% of the customers being female and 40% of the customers being male. 

In [83]:
#Find out the what demographic placed the most orders
conn = sqlite3.connect('customer.db')
sql_query = """
SELECT 
    CASE
        WHEN age BETWEEN 0 AND 19 THEN '0-19'
        WHEN age BETWEEN 20 AND 29 THEN '20-29'
        WHEN age BETWEEN 30 AND 39 THEN '30-39'
        WHEN age BETWEEN 40 AND 49 THEN '40-49'
        WHEN age BETWEEN 50 AND 59 THEN '50-59'
        ELSE '60+'
    END AS Age_Bracket,
    COUNT(*) AS Total_Customers,
    gender
FROM customer
GROUP BY Age_Bracket, gender
ORDER BY Total_Customers DESC
LIMIT 5
;"""

sql_query = pd.read_sql_query(sql_query, conn)
conn.close()
print(sql_query)

  Age_Bracket  Total_Customers  gender
0       30-39            11507  Female
1       40-49            11504  Female
2       20-29            11503  Female
3         60+            11459  Female
4       50-59            11301  Female


Females from all ages 20-60+ were the most common customers demographic.  This is important when deciding on marketing techniques.  

The marketing should focus on females but should not be age specific or targeted at one age group. 

In [92]:
#Find the most common transactions by age group, gender, and payment method. 
conn = sqlite3.connect('customer.db')
sql_query = """
SELECT 
    CASE
        WHEN age BETWEEN 0 AND 19 THEN '0-19'
        WHEN age BETWEEN 20 AND 29 THEN '20-29'
        WHEN age BETWEEN 30 AND 39 THEN '30-39'
        WHEN age BETWEEN 40 AND 49 THEN '40-49'
        WHEN age BETWEEN 50 AND 59 THEN '50-59'
        ELSE '60+'
    END AS Age_Bracket,
    COUNT(*) AS Total_Customers,
    gender,
    payment_method
FROM customer
GROUP BY Age_Bracket, gender, payment_method
ORDER BY Total_Customers DESC
LIMIT 10
;"""

sql_query = pd.read_sql_query(sql_query, conn)
conn.close()
print(sql_query)

  Age_Bracket  Total_Customers  gender payment_method
0         60+             5193  Female           Cash
1       20-29             5141  Female           Cash
2       30-39             5083  Female           Cash
3       40-49             5081  Female           Cash
4       50-59             5039  Female           Cash
5       30-39             4116  Female    Credit Card
6       40-49             4101  Female    Credit Card
7       20-29             4008  Female    Credit Card
8         60+             3993  Female    Credit Card
9       50-59             3988  Female    Credit Card


The most common transcations were females aged 20-60+ using cash as the payment method.  The second most common was females using credit cards.  

<H1> Key Insights </H1>

- The company was mainly doing transactions with female customers.
- There was little variance in the age group from 20 years old to 60+.
- Most of the transactions were done using cash (44%) and credit cards making up 35% of the sales.
- Although 40% of transactions came from males, females were the most demographic when grouped by age group and payment method. 
- The company should focus their marketing on female customers without being age specific. 